In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [7]:
X= pd.read_csv("../data/X_processed.csv", index_col=0)
y = pd.read_csv("../data/y_labels.csv", index_col=0).squeeze()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Scale AFTER split to prevent data leakage
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      index=X_test.index,  columns=X_test.columns)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Train label distribution:\n{y_train.value_counts()}")

In [ ]:
# Logistic Regression with L2 regularization
lr = LogisticRegression(max_iter=1000, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring="accuracy")
print(f"LR CV accuracy: {lr_cv_scores.mean():.3f} ± {lr_cv_scores.std():.3f}")

lr.fit(X_train_scaled, y_train)
print("\nTest set report (Logistic Regression):")
print(classification_report(y_test, lr.predict(X_test_scaled), target_names=["SCC", "Adeno"]))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)

rf_cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring="accuracy")
print(f"RF CV accuracy: {rf_cv_scores.mean():.3f} ± {rf_cv_scores.std():.3f}")

rf.fit(X_train_scaled, y_train)
print("\nTest set report (Random Forest):")
print(classification_report(y_test, rf.predict(X_test_scaled), target_names=["SCC", "Adeno"]))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, model, name in zip(axes, [lr, rf], ["Logistic Regression", "Random Forest"]):
    cm = confusion_matrix(y_test, model.predict(X_test_scaled))
    ConfusionMatrixDisplay(cm, display_labels=["SCC", "Adeno"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("../figures/confusion_matrices.png", dpi=150)
plt.show()